In [16]:
import math
import numpy as np

def calc_aim_error(obs):    
   
    best_dist = 1e9
    best_x = best_y = best_z = None
    base = 6
    for i in range(15):
        j = base + i * 8
        alive = obs[j + 6] > 0.5
        if alive:
            x = obs[j] * 100.0
            z = obs[j + 1] * 100.0
            y = obs[j + 5] * 20.0
            dist = math.sqrt(x * x + z * z)
            if dist < best_dist:
                best_dist = dist
                best_x, best_y, best_z = x, y, z
    if best_x is None:
        return None
    cur_yaw = obs[3] * math.pi
    cur_pitch = obs[4] * 0.9
    ideal_yaw = math.atan2(best_z, best_x)
    xz_dist = max(math.sqrt(best_x**2 + best_z**2), 0.5)
    ideal_pitch = math.atan2(best_y - 2.3, xz_dist)
    yaw_err = abs(wrap_angle(ideal_yaw - cur_yaw))
    pitch_err = abs(ideal_pitch - cur_pitch)
    return yaw_err + pitch_err

def wrap_angle(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

In [20]:
class FullRewardWrapper(gym.Wrapper):
    def __init__(self, env, failure_states, success_obs, success_act,
                 penalty_threshold=0.3, sil_threshold=0.1, sil_bonus=0.2):
        super().__init__(env)
        self.failure_states = np.array(failure_states)
        self.success_obs = np.array(success_obs)
        self.success_act = np.array(success_act).astype(int)
        self.penalty_threshold = penalty_threshold
        self.sil_threshold = sil_threshold
        self.sil_bonus = sil_bonus
        self.prev_score = 0
        self.kills = 0
        self._steady_count = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_score = info["hunterScore"]
        self.kills = 0
        self._steady_count = 0
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        shaped = reward
        score_delta = info["hunterScore"] - self.prev_score
        self.prev_score = info["hunterScore"]

        # Kill bonus
        if score_delta > 0:
            shaped += score_delta * 5.0
            self.kills += 1

        # Aim bonus
        aim_err = self._calc_aim_error(obs)
        if aim_err is not None:
            shaped += 2.0 * np.exp(-aim_err**2 / 0.1)
            if aim_err < 0.08:
                shaped += 2.0
            if action == 1:
                if aim_err < 0.15:
                    shaped += 1.5
                else:
                    shaped -= 1.2

        # Tree penalty
        if action == 1 and self._is_tree_blocked(obs):
            shaped -= 2.0

        # Wasteful shot penalty
        if action == 1 and self._count_alive(obs) == 0:
            shaped -= 2.0

        # Failure penalty
        if len(self.failure_states) > 0:
            dists = np.linalg.norm(self.failure_states - obs[:169], axis=1)
            if np.min(dists) < self.penalty_threshold:
                shaped -= 1.0

        # SIL bonus
        if len(self.success_obs) > 0:
            # Check whether the current observation is close to a success state
            # and whether the selected action matches the corresponding success action.
            # For efficiency, random sampling or KDTree can be used.
            dists = np.linalg.norm(self.success_obs - obs[:169], axis=1)
            close_indices = np.where(dists < self.sil_threshold)[0]
            if len(close_indices) > 0:
                # Check whether the executed action matches any recorded success action.
                matching_actions = (self.success_act[close_indices] == action)
                if np.any(matching_actions):
                    shaped += self.sil_bonus

        info["kills"] = self.kills
        return obs, shaped, terminated, truncated, info

    def calc_aim_error(obs):    
   
        best_dist = 1e9
        best_x = best_y = best_z = None
        base = 6
        for i in range(15):
            j = base + i * 8
            alive = obs[j + 6] > 0.5
            if alive:
                x = obs[j] * 100.0
                z = obs[j + 1] * 100.0
                y = obs[j + 5] * 20.0
                dist = math.sqrt(x * x + z * z)
                if dist < best_dist:
                    best_dist = dist
                    best_x, best_y, best_z = x, y, z
        if best_x is None:
            return None
        cur_yaw = obs[3] * math.pi
        cur_pitch = obs[4] * 0.9
        ideal_yaw = math.atan2(best_z, best_x)
        xz_dist = max(math.sqrt(best_x**2 + best_z**2), 0.5)
        ideal_pitch = math.atan2(best_y - 2.3, xz_dist)
        yaw_err = abs(wrap_angle(ideal_yaw - cur_yaw))
        pitch_err = abs(ideal_pitch - cur_pitch)
        return yaw_err + pitch_err
        
     def _is_tree_blocked(self, obs):
       
        if not self._trees:
            return False
        yaw = obs[3] * math.pi
        pitch = obs[4] * 0.9
        cp = math.cos(pitch)
        dx = math.cos(yaw) * cp
        dy = math.sin(pitch)
        dz = math.sin(yaw) * cp
        bx, by, bz = dx*3.0, 2.3 + dy*3.0, dz*3.0
        d = 0.0
        while d < 90.0:
            bx += dx * 3.0
            by += dy * 3.0
            bz += dz * 3.0
            d += 3.0
            if by < -2.0 or by > 80.0:
                break
            for t in self._trees:
                tx, tz, tr = t["x"], t["z"], t.get("r", 2.5)
                if (bx - tx)**2 + (bz - tz)**2 < (tr + 0.3)**2:
                    return True
        return False
        
    def _count_alive(self, obs):
        return sum(1 for i in range(15) if obs[6+i*8+6] > 0.5)

In [21]:
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO
import shooter

def collect_success_data(model_path, num_episodes=50):
    model = PPO.load(model_path)
    env = gym.make("Shooter-v0", render_mode=None)
    
    success_obs = []
    success_act = []
    steady_count = 0
    steady_buffer_obs = []
    steady_buffer_act = []
    
    for ep in range(num_episodes):
        obs, info = env.reset()
        prev_score = info["hunterScore"]
        prev_obs = obs.copy()
        done = False
        
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs_next, reward, terminated, truncated, info = env.step(action)
            
            aim_err = calc_aim_error(obs)  
            
            
            if aim_err is not None and aim_err < 0.08:
                steady_count += 1
                steady_buffer_obs.append(obs.copy())
                steady_buffer_act.append(action)
                if steady_count >= 3:
                    success_obs.extend(steady_buffer_obs)
                    success_act.extend(steady_buffer_act)
                    steady_buffer_obs.clear()
                    steady_buffer_act.clear()
                    steady_count = 0
            else:
                steady_count = 0
                steady_buffer_obs.clear()
                steady_buffer_act.clear()
            
            
            score_delta = info["hunterScore"] - prev_score
            if score_delta > 0 and action == 1:
                success_obs.append(prev_obs)
                success_act.append(1)
            
            prev_score = info["hunterScore"]
            prev_obs = obs.copy()
            obs = obs_next
            
            if terminated or truncated:
                break
    
    env.close()
    return np.array(success_obs), np.array(success_act)

In [18]:
success_obs, success_act = collect_success_data("ppo_v9_failure_finetuned", num_episodes=50)
np.save("success_obs.npy", success_obs)
np.save("success_act.npy", success_act)
print(f"Collected {len(success_obs)} success transitions")

Collected 10450 success transitions


In [19]:
def collect_failure_states_from_model(model_path, num_episodes=100):
    model = PPO.load(model_path)
    env = gym.make("Shooter-v0", render_mode=None)
    failure_states = []
    
    for ep in range(num_episodes):
        obs, info = env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs_next, reward, terminated, truncated, info = env.step(action)
            
            aim_err = calc_aim_error(obs)
            if action == 1 and (aim_err is None or aim_err > 0.15):
                failure_states.append(obs.copy())
            
            obs = obs_next
            if terminated:
                # เก็บ state ก่อน game over (obs ก่อนจะหมด episode)
                failure_states.append(obs.copy())
                break
            if truncated:
                break
    env.close()
    return np.array(failure_states)

new_failures = collect_failure_states_from_model("ppo_v9_failure_finetuned", num_episodes=100)


try:
    old_failures = np.load("failure_states.npy")
    combined = np.concatenate([old_failures, new_failures], axis=0)
except:
    combined = new_failures
np.save("failure_states_v2.npy", combined)
print(f"Total failure states: {len(combined)}")

Total failure states: 1588
